> **Production note (2026-06-21):** The script pipeline in `scripts/` is the source of truth for final outputs. This notebook is retained for exploration and narrative context; run the README pipeline for reproducible delivery artifacts.


# Fuel Price Features — Exploratory Analysis & Feature Engineering
## Repsol Capstone Project — Sprint 2 (Extension)

**Goal:** Process daily provincial fuel prices (2023–2025), aggregate to monthly national + regional level,
explore the price ↔ biodiesel demand relationship, and generate price features for model retraining.

**Inputs:**
- `data/inputs/precios_combustibles_2023/24/25.csv` — daily price per province × product (PAI + PVP)
- `data/inputs/consumo_biodiesel_targets.csv` — monthly biodiesel demand per target region

**Outputs:**
- `data/features/features_precios_combustibles.csv` — monthly price features (national + regional, all products)

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

NOTEBOOK_DIR  = Path().resolve()
REPO_ROOT     = NOTEBOOK_DIR.parent
DATA_INPUTS   = REPO_ROOT / 'data' / 'inputs'
DATA_FEATURES = REPO_ROOT / 'data' / 'features'
FIGS          = REPO_ROOT / 'reports' / 'figures'

# Province → Region mapping (only 4 target regions + Nacional)
REGION_MAP = {
    'Madrid':              'Madrid',
    'Barcelona':           'Cataluña',
    'Girona':              'Cataluña',
    'Lleida':              'Cataluña',
    'Tarragona':           'Cataluña',
    'Almería':             'Andalucía',
    'Cádiz':               'Andalucía',
    'Córdoba':             'Andalucía',
    'Granada':             'Andalucía',
    'Huelva':              'Andalucía',
    'Jaén':                'Andalucía',
    'Málaga':              'Andalucía',
    'Sevilla':             'Andalucía',
    'Alicante/Alacant':    'Valencia',
    'Castellón/Castelló':  'Valencia',
    'Valencia/València':   'Valencia',
}

# Short names for the 4 fuel products
PRODUCT_MAP = {
    'Gasolina 95 E5':    'gasolina95',
    'Gasolina 98 E5':    'gasolina98',
    'Gasóleo A habitual':'gasoleo_a',
    'Gasóleo Premium':   'gasoleo_prem',
}

print('Setup complete.')

## 1. Load & Clean Raw Price Data

### What?
Combine 3 annual CSV files. The files use semicolon separators and comma decimal notation.

### Why?
Daily granularity (~75k rows/year × 3 years = ~227k rows) will be aggregated to monthly averages.
We need both PAI (cost price) and PVP (consumer price) as they capture different economic signals.

In [ ]:
frames = []
for yr in [2023, 2024, 2025]:
    df = pd.read_csv(DATA_INPUTS   / f'precios_combustibles_{yr}.csv', sep=';', decimal=',')
    frames.append(df)
    print(f'{yr}: {df.shape[0]:,} rows')

df_raw = pd.concat(frames, ignore_index=True)

# Rename columns to clean short names
df_raw.columns = ['Fecha', 'Province', 'Product', 'PAI', 'PVP']
df_raw['Fecha'] = pd.to_datetime(df_raw['Fecha'])
df_raw['Product_short'] = df_raw['Product'].map(PRODUCT_MAP)

print(f'\nTotal rows: {len(df_raw):,}')
print(f'Date range: {df_raw["Fecha"].min().date()} → {df_raw["Fecha"].max().date()}')
print(f'\nNull check:')
print(df_raw[['PAI','PVP']].isnull().sum())
df_raw.head(3)

## 2. Monthly Aggregation — National Level

### What?
Average daily prices across all 52 provinces → monthly national price per product and price type.

### Why?
Our biodiesel demand model is at monthly frequency. The national price is the simple mean across
all provincial observations for each month (equal-weighted — we don't have volume weights).

In [ ]:
df_raw['Fecha_mes'] = df_raw['Fecha'].dt.to_period('M').astype(str)

# National monthly mean (all provinces)
df_nac = (
    df_raw
    .groupby(['Fecha_mes', 'Product_short'])[['PAI', 'PVP']]
    .mean()
    .reset_index()
)

# Pivot: one row per month, one column per (product × price_type)
df_nac_wide = df_nac.pivot(index='Fecha_mes', columns='Product_short', values=['PAI','PVP'])
df_nac_wide.columns = [f'{pt}_{prod}_nac' for pt, prod in df_nac_wide.columns]
df_nac_wide = df_nac_wide.reset_index().rename(columns={'Fecha_mes': 'Fecha'})

print(f'Nacional monthly shape: {df_nac_wide.shape}')
print('Columns:', df_nac_wide.columns.tolist())
df_nac_wide.round(4).head()

## 3. Monthly Aggregation — Regional Level

### What?
Average daily prices for the provinces belonging to each of the 4 target regions.

### Why?
Regional fuel prices vary — Madrid has the lowest diesel taxes, Cataluña and Valencia differ.
Using region-specific prices gives the ML models a more precise demand signal per target.

In [ ]:
df_reg_src = df_raw.copy()
df_reg_src['Region'] = df_reg_src['Province'].map(REGION_MAP)
df_reg_src = df_reg_src.dropna(subset=['Region'])  # keep only our 4 regions

df_reg = (
    df_reg_src
    .groupby(['Fecha_mes', 'Region', 'Product_short'])[['PAI', 'PVP']]
    .mean()
    .reset_index()
)

regional_frames = []
for region in ['Madrid', 'Cataluña', 'Andalucía', 'Valencia']:
    reg_slug = region.lower().replace('ú','u').replace('ñ','n').replace('á','a')
    sub = df_reg[df_reg['Region'] == region].copy()
    wide = sub.pivot(index='Fecha_mes', columns='Product_short', values=['PAI','PVP'])
    wide.columns = [f'{pt}_{prod}_{reg_slug}' for pt, prod in wide.columns]
    wide = wide.reset_index().rename(columns={'Fecha_mes': 'Fecha'})
    regional_frames.append(wide)
    print(f'{region}: {wide.shape}')

# Merge all regions
from functools import reduce
df_reg_wide = reduce(lambda a, b: a.merge(b, on='Fecha', how='outer'), regional_frames)
print(f'\nAll regions merged: {df_reg_wide.shape}')
df_reg_wide.round(4).head(3)

## 4. Combine National + Regional → price_features.csv

In [ ]:
df_prices = df_nac_wide.merge(df_reg_wide, on='Fecha', how='outer').sort_values('Fecha').reset_index(drop=True)

print(f'Price features shape: {df_prices.shape}')
print(f'Date range: {df_prices["Fecha"].iloc[0]} → {df_prices["Fecha"].iloc[-1]}')
print(f'Null count total: {df_prices.isnull().sum().sum()}')
df_prices.head()

## 5. Exploratory Analysis — Price Trends

### What?
Visualise how the 4 fuel prices evolved over 2023–2025 at national level.

### Why?
Before testing correlation with biodiesel demand, we need to understand the price dynamics:
are prices trending? seasonal? volatile? This contextualises the feature's predictive value.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

products = ['gasoleo_a', 'gasoleo_prem', 'gasolina95', 'gasolina98']
labels   = ['Diesel A habitual', 'Diesel Premium', 'Gasoline 95 E5', 'Gasoline 98 E5']
colors   = ['#2C7BB6', '#D7191C', '#1A936F', '#FF6B35']

for price_type, ax in zip(['PVP', 'PAI'], axes):
    for prod, lbl, col in zip(products, labels, colors):
        col_name = f'{price_type}_{prod}_nac'
        if col_name not in df_prices.columns:
            continue
        ax.plot(df_prices['Fecha'], df_prices[col_name], color=col, linewidth=2,
                marker='o', markersize=4, label=lbl)
    ax.set_title(f'Price {price_type} (€/litro) — Media Nacional 2023–2025', fontweight='bold')
    ax.set_ylabel('€/litro')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIGS / '12_price_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 12_price_trends.png')

## 6. Price ↔ Demand Correlation Analysis

### What?
Test the correlation between monthly fuel prices and monthly biodiesel demand (Nacional + 4 targets).
Include both contemporaneous correlation and lagged correlation (price at t-1 predicting demand at t).

### Why?
If price at t-1 correlates with demand at t, it's a usable lag feature (we know last month's price
when forecasting this month). Contemporaneous correlation is informative but not directly usable
in a real-time forecasting scenario.

In [ ]:
master = pd.read_csv(DATA_INPUTS / 'master_dataset.csv')

TARGET_LABEL = {
    'ESPAÑA':               'Nacional',
    'Andalucía':            'Andalucía',
    'Cataluña':             'Cataluña',
    'Madrid, Comunidad de': 'Madrid',
    'Comunitat Valenciana': 'Valencia',
}

df_demand = (
    master[master['Target'] == 1][['Fecha', 'CCAA', 'Consumo_Tm']]
    .copy()
    .assign(Target=lambda d: d['CCAA'].map(TARGET_LABEL))
)

# Merge Nacional demand with national prices
df_nac_demand = df_demand[df_demand['Target'] == 'Nacional'][['Fecha', 'Consumo_Tm']].copy()
df_merged = df_nac_demand.merge(df_prices, on='Fecha', how='inner')

print(f'Merged shape (Nacional × prices): {df_merged.shape}')

price_cols = [c for c in df_prices.columns if c != 'Fecha' and '_nac' in c]

# Full-window correlation (2023-2025) -- DIAGNOSTIC ONLY. Computing this over
# the full window (which includes the 2025 test period) and then using it to
# decide whether to retrain models on price features would mean the test set
# influenced a feature-selection decision -- the same leakage risk as picking
# a model family by test-set MAPE. Shown here for context, not as the decision input.
corr_0 = df_merged[price_cols + ['Consumo_Tm']].corr()['Consumo_Tm'].drop('Consumo_Tm').rename('corr_t0')
corr_1 = {}
for c in price_cols:
    shifted = df_merged[c].shift(1)
    corr_1[c] = df_merged['Consumo_Tm'].corr(shifted)
corr_1 = pd.Series(corr_1, name='corr_t-1')
df_corr = pd.concat([corr_0, corr_1], axis=1).sort_values('corr_t0', ascending=False)

# Train-only correlation (2023-2024) -- the actual decision input (see Section 10).
TRAIN_END = '2024-12'
df_merged_train = df_merged[df_merged['Fecha'] <= TRAIN_END].copy()
corr_0_train = (
    df_merged_train[price_cols + ['Consumo_Tm']].corr()['Consumo_Tm']
    .drop('Consumo_Tm').rename('corr_t0_train')
)
corr_1_train = {}
for c in price_cols:
    shifted = df_merged_train[c].shift(1)
    corr_1_train[c] = df_merged_train['Consumo_Tm'].corr(shifted)
corr_1_train = pd.Series(corr_1_train, name='corr_t-1_train')
df_corr_train = pd.concat([corr_0_train, corr_1_train], axis=1).sort_values('corr_t0_train', ascending=False)

print('\nFull-window (2023-2025) correlation with Nacional biodiesel demand -- DIAGNOSTIC ONLY:')
print(df_corr.round(3).to_string())
print('\nTrain-only (2023-2024) correlation with Nacional biodiesel demand -- DECISION INPUT:')
print(df_corr_train.round(3).to_string())

In [ ]:
# Visual: scatter plots of top 4 most correlated price features
top4 = df_corr['corr_t0'].abs().nlargest(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, col in zip(axes, top4):
    ax.scatter(df_merged[col], df_merged['Consumo_Tm'],
               color='#004E89', alpha=0.7, edgecolors='white', s=60)
    corr_v = df_merged[col].corr(df_merged['Consumo_Tm'])
    # Trend line
    z = np.polyfit(df_merged[col].dropna(), df_merged.loc[df_merged[col].notna(), 'Consumo_Tm'], 1)
    xfit = np.linspace(df_merged[col].min(), df_merged[col].max(), 50)
    ax.plot(xfit, np.poly1d(z)(xfit), color='#D7191C', linewidth=2)
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel('Consumption Biodiesel (Tm)')
    ax.set_title(f'r = {corr_v:.3f}', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Price ↔ Biodiesel Demand — Top 4 Correlations (Nacional)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGS / '13_price_demand_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 13_price_demand_scatter.png')

In [ ]:
# Dual-axis: price PVP Diesel A vs Consumption Biodiesel over time
fig, ax1 = plt.subplots(figsize=(16, 5))

color_demand = '#004E89'
color_price  = '#D7191C'

ax1.bar(range(len(df_merged)), df_merged['Consumo_Tm'],
        color=color_demand, alpha=0.5, label='Consumption Biodiesel (Tm)')
ax1.set_ylabel('Consumption Biodiesel (Tm)', color=color_demand)
ax1.tick_params(axis='y', labelcolor=color_demand)
ax1.set_xticks(range(len(df_merged)))
ax1.set_xticklabels(df_merged['Fecha'].tolist(), rotation=45, ha='right', fontsize=8)

ax2 = ax1.twinx()
pvp_col = 'PVP_gasoleo_a_nac'
if pvp_col in df_merged.columns:
    ax2.plot(range(len(df_merged)), df_merged[pvp_col],
             color=color_price, linewidth=2.5, marker='o', markersize=4,
             label='PVP Diesel A (€/L)')
    ax2.set_ylabel('PVP Diesel A (€/litro)', color=color_price)
    ax2.tick_params(axis='y', labelcolor=color_price)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.set_title('Price PVP Diesel A vs Consumption Biodiesel Diesel Nexa — Nacional 2023–2025',
              fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGS / '14_price_vs_demand_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 14_price_vs_demand_timeline.png')

## 7. Correlation Heatmap — All Targets × Price Features

In [ ]:
TARGET_REGION_PRICE = {
    'Nacional':  '_nac',
    'Madrid':    '_madrid',
    'Cataluña':  '_cataluna',
    'Andalucía': '_andalucia',
    'Valencia':  '_valencia',
}

heatmap_data = {}

for tgt, suffix in TARGET_REGION_PRICE.items():
    demand = df_demand[df_demand['Target'] == tgt][['Fecha','Consumo_Tm']]
    merged = demand.merge(df_prices, on='Fecha', how='inner')
    price_cols_tgt = [c for c in df_prices.columns if c.endswith(suffix)]
    for pc in price_cols_tgt:
        if pc in merged.columns:
            r = merged['Consumo_Tm'].corr(merged[pc])
            short_name = pc.replace(suffix, '')
            heatmap_data.setdefault(tgt, {})[short_name] = round(r, 3)

df_heat = pd.DataFrame(heatmap_data).T
print('Correlation matrix (demand × price features):')
print(df_heat.to_string())

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(df_heat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Correlation: Biodiesel Demand vs Fuel Price Features (contemporaneous)',
             fontweight='bold')
ax.set_xlabel('Price Feature')
ax.set_ylabel('Target Region')
plt.tight_layout()
plt.savefig(FIGS / '15_price_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 15_price_correlation_heatmap.png')

## 8. Select Model Features & Add Lag_1

### What?
From the full price matrix, select the features that will be added to the ML model dataset.
We add **lag_1** versions (price at t-1) so they are available at prediction time.

### Why?
A contemporaneous price feature (price at the same month as demand) would require knowing
the price before we observe the demand — valid for modeling but requires price forecasts
in the 24-month forward horizon. A lag_1 feature is causally clean: we always know last
month's price when making a forecast.

In [ ]:
# Keep: national prices for all 4 products × both PAI/PVP + regional PVP Diesel A
keep_cols = ['Fecha'] + [c for c in df_prices.columns if c != 'Fecha']

df_feat = df_prices[keep_cols].copy().sort_values('Fecha').reset_index(drop=True)

# Add lag_1 for all price columns
price_feature_cols = [c for c in keep_cols if c != 'Fecha']
for c in price_feature_cols:
    df_feat[f'{c}_lag1'] = df_feat[c].shift(1)

print(f'Price features table shape: {df_feat.shape}')
print(f'Columns ({len(df_feat.columns)}): {df_feat.columns.tolist()[:10]} ...')
print(f'\nNull count per column (should be 1 lag row):')
print(df_feat.isnull().sum()[df_feat.isnull().sum() > 0])

## 9. Save

In [ ]:
out = DATA_FEATURES / 'features_precios_combustibles.csv'
df_feat.to_csv(out, index=False, encoding='utf-8')

verify = pd.read_csv(out)
print(f'Saved: {out.name}  →  {verify.shape[0]} rows × {verify.shape[1]} cols')
print(verify.head(3).to_string())

## 10. Summary — Should We Retrain?

### Decision criteria
We proceed to retrain the ML models with price features if, using **train-only (2023-2024) correlation**:
- At least one price feature shows |r| > 0.40 with demand (moderate correlation)
- The lag_1 version also shows |r| > 0.30 (usable for forecasting)

The decision deliberately uses the train-only correlation (`df_corr_train`), not the full-window
correlation that includes 2025. Using the full window would let the test period influence a
feature-selection decision before any model is even trained on it -- the same leakage risk as
picking a model family by test-set MAPE. The full-window numbers are still shown above for
context, since they describe the relationship across the whole observed history, but they are
not the gate.

SARIMA does not use price features (SARIMAX with future exogenous values would require
forecasting prices separately — added complexity without clear gain at this sample size).

In [ ]:
print('='*65)
print('PRICE ↔ DEMAND CORRELATION SUMMARY (Nacional)')
print('='*65)
print('\nFull-window (2023-2025), diagnostic only:')
print(df_corr.round(3).to_string())
print('\nTrain-only (2023-2024), decision input:')
print(df_corr_train.round(3).to_string())

strong_t0 = df_corr_train[df_corr_train['corr_t0_train'].abs() > 0.40]
strong_lag = df_corr_train[df_corr_train['corr_t-1_train'].abs() > 0.30]

print(f'\n[Train-only] Features with |r_t0| > 0.40: {len(strong_t0)}')
print(f'[Train-only] Features with |r_lag1| > 0.30: {len(strong_lag)}')

if len(strong_t0) > 0 or len(strong_lag) > 0:
    print('\n✅ RECOMMENDATION: Proceed to 08_modeling_with_prices.ipynb')
    print('   Price features show meaningful train-only correlation — retraining RF + XGBoost is justified.')
    print('   Note: 08_modeling_with_prices.ipynb then re-checks this with train-only walk-forward CV,')
    print('   which is the stronger test (predictive performance, not just correlation).')
else:
    print('\n⚠️  Weak train-only correlation — price features may not improve ML models significantly.')
    print('   Still worth testing in 08_modeling_with_prices.ipynb (small dataset: correlation != predictive power).')

print('\n' + '='*65)
print('OUTPUT FILES')
print('='*65)
print(f'  data/features/features_precios_combustibles.csv     {verify.shape}')
print('  reports/figures/12_price_trends.png')
print('  reports/figures/13_price_demand_scatter.png')
print('  reports/figures/14_price_vs_demand_timeline.png')
print('  reports/figures/15_price_correlation_heatmap.png')